In [1]:
import torch
import numpy as np
import pandas as pd
from haversine import haversine, Unit
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder, StandardScaler


/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 100

# 1. Load Dataset

In [3]:
trainpath = f'../../../data/top30groups/LongLatCombined/train1/train{partition}.csv'
testpath = f'../../../data/top30groups/LongLatCombined/test1/test{partition}.csv'
traindata = pd.read_csv(trainpath, encoding='ISO-8859-1')
testdata = pd.read_csv(testpath, encoding='ISO-8859-1')

In [4]:
combined = pd.concat([traindata, testdata], axis = 0)

### Find unique locations and construct global graph

In [5]:
# Extract unique locations for node creation
combined['location'] = list(zip(combined['longitude'], combined['latitude']))
unique_locations = combined['location'].drop_duplicates().reset_index(drop=True)

print(combined['location'].nunique())
print(len(unique_locations))
# Map locations to an identity
location2id = {loc: idx for idx, loc in enumerate(unique_locations)}
combined['location_id'] = combined['location'].map(location2id)

# Encode labels
le = LabelEncoder()
combined['label'] = le.fit_transform(combined['gname'])

# Aggregate features per location (only the 3 you want)
grouped = combined.groupby('location')
agg_df = grouped.agg({
    'nkill': 'mean',
    'attacktype1': lambda x: x.value_counts().idxmax(),
    'target1': lambda x: x.value_counts().idxmax(),
}).reset_index()

# Rename columns (optional but clean)
agg_df.columns = [
    'location',
    'nkill_mean',
    'attacktype_code',
    'targettype_code'
]

# Merge with coordinates
coords_df = pd.DataFrame(unique_locations.tolist(), columns=['longitude', 'latitude'])
coords_df['location'] = list(zip(coords_df['longitude'], coords_df['latitude']))
features_df = pd.merge(coords_df, agg_df, on='location', how='left')
features_df = features_df.fillna(0)

# Select features
feature_columns = [
    'longitude', 'latitude',  # original spatial info
    'nkill_mean',
    'attacktype_code',
    'targettype_code'
]

print(len(coords_df))

# Standardize features and convert to tensor
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_global = scaler.fit_transform(features_df[feature_columns])
x_global = torch.tensor(x_global, dtype=torch.float)

print("Feature Matrix shape: ", x_global.shape)

# Build global edge list using 1km Haversine
from haversine import haversine, Unit

edges = []
coords_latlon = [(lat, lon) for lon, lat in unique_locations]
for i in range(len(coords_latlon)):
    for j in range(i + 1, len(coords_latlon)):
        if haversine(coords_latlon[i], coords_latlon[j], Unit.KILOMETERS) <= 1.0:
            edges.append((i, j))
            edges.append((j, i))

global_edge_index = torch.tensor(edges, dtype=torch.long).T


1790
1790
1790
Feature Matrix shape:  torch.Size([1790, 5])


In [6]:
print(features_df.iloc[0])


longitude                        -76.978462
latitude                         -11.967368
location           (-76.978462, -11.967368)
nkill_mean                         0.210084
attacktype_code                           3
targettype_code                         399
Name: 0, dtype: object


In [7]:
global_edge_index.shape

torch.Size([2, 242])

In [8]:
unique_nodes = torch.unique(global_edge_index)
print("Nodes with at least one neighbor: ", len(unique_nodes))

Nodes with at least one neighbor:  161


### Creating subgraphs for each node depending on its neighbors

In [9]:
def get_subgraph(center_id, edge_index, x_global):
    # Get neighbors (indices) of center node
    neighbors = edge_index[1][edge_index[0] == center_id]
    node_ids = torch.cat([torch.tensor([center_id]), neighbors]).unique()

    # Remap node indices locally
    id_map = {old_id.item(): i for i, old_id in enumerate(node_ids)}
    new_edges = []
    for source, destination in zip(*edge_index):
        if source in node_ids and destination in node_ids:
            new_edges.append((id_map[source.item()], id_map[destination.item()]))

    # If no edges exist, add a self-loop
    if len(new_edges) == 0:
        center_local_idx = 0  # only node in subgraph
        new_edges = [(0, 0)]
    else:
        center_local_idx = id_map[center_id.item()]

    sub_x = x_global[node_ids]
    sub_edge_index = torch.tensor(new_edges).T

    return sub_x, sub_edge_index, center_local_idx


### Subgraphs for train

In [10]:
from torch_geometric.data import Data

traindata_list = []
for _, row in traindata.iterrows():
    center_id = location2id[(row['longitude'], row['latitude'])]
    label = le.transform([row['gname']])[0]
    
    x, edge_index, center_idx = get_subgraph(torch.tensor(center_id), global_edge_index, torch.tensor(x_global, dtype=torch.float))
    
    traindata_obj = Data(x=x, edge_index=edge_index, y=torch.tensor(label), center=center_idx)
    traindata_list.append(traindata_obj)


/tmp/ipykernel_112143/4138672406.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x, edge_index, center_idx = get_subgraph(torch.tensor(center_id), global_edge_index, torch.tensor(x_global, dtype=torch.float))
/tmp/ipykernel_112143/4138672406.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x, edge_index, center_idx = get_subgraph(torch.tensor(center_id), global_edge_index, torch.tensor(x_global, dtype=torch.float))
/tmp/ipykernel_112143/4138672406.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x, edge_index, center_idx = get_s

### Subgraphs for test

In [11]:
test_data_list = []
for _, row in testdata.iterrows():
    loc = (row['longitude'], row['latitude'])
    
    # Skip if location not in mapping (just in case)
    if loc not in location2id:
        continue
    
    center_id = location2id[loc]
    label = le.transform([row['gname']])[0]
    
    x, edge_index, center_idx = get_subgraph(
        torch.tensor(center_id),
        global_edge_index,
        torch.tensor(x_global, dtype=torch.float)
    )

    testdata_obj = Data(x=x, edge_index=edge_index, y=torch.tensor(label), center=center_idx)
    test_data_list.append(testdata_obj)

/tmp/ipykernel_112143/2490495565.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(x_global, dtype=torch.float)
/tmp/ipykernel_112143/2490495565.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(x_global, dtype=torch.float)
/tmp/ipykernel_112143/2490495565.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(x_global, dtype=torch.float)
/tmp/ipykernel_112143/2490495565.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clo

### GCN Model

In [12]:
import torch.nn as nn


class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, activation_fn=F.relu):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.classifier = nn.Linear(hidden_channels, out_channels)
        self.activation_fn = activation_fn

    def forward(self, batch):
        x, edge_index = batch.x, batch.edge_index
        #x = self.conv1(x, edge_index)
        #x = F.relu(x)
        x = self.activation_fn(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)

        # batch.ptr[:-1] extracts central node index
        center_embeddings = x[batch.ptr[:-1]]
        out = self.classifier(center_embeddings)
        return F.log_softmax(out, dim=1)

### Training and Testing

In [13]:
from sklearn.model_selection import train_test_split

train_set, val_set = train_test_split(traindata_list, test_size=0.2, random_state=42)


### Define batches

In [14]:
def train(loader):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    for batch in loader:
        batch = batch.to(device)
        out = model(batch)
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()
        total += batch.y.size(0)
    return correct / total

In [15]:
from itertools import product

best_model_state = None
best_val_acc = 0.0
patience = 100

import torch.nn.functional as F

activation_map = {
    'relu': F.relu,
    'tanh': F.tanh
}

param_dist = {
    'h1': [10, 50, 100, 150, 200, 300],
    'activations': ['relu', 'tanh'],
    'lrs': [0.0001, 0.001, 0.01],
    #'alphas': [1e-5, 1e-4, 1e-3, 1e-2]
    'batch_sizes': [128, 256, 512]
    }

keys, values = zip(*param_dist.items())
combinations = [dict(zip(keys, v)) for v in product(*values)]

import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

best_params = None
num_classes = len(le.classes_)

for i, params in enumerate(combinations):
    print(f"Combination {i+1}/{len(combinations)}: {params}")

    act_fn = activation_map[params["activations"]]

    model = GCN(in_channels=5, hidden_channels=params["h1"], out_channels=len(le.classes_), activation_fn = act_fn).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lrs"], weight_decay=5e-4)
    criterion = torch.nn.CrossEntropyLoss()

    train_loader = DataLoader(train_set, batch_size=params["batch_sizes"], shuffle=True)
    val_loader = DataLoader(val_set, batch_size=params["batch_sizes"])
    test_loader = DataLoader(test_data_list, batch_size=params["batch_sizes"])

    patience_counter = 0

    for epoch in range(1, 1000):
        avg_loss = train(train_loader)

        train_acc = evaluate(train_loader)
        val_acc = evaluate(val_loader)

        if epoch % 50 == 0:
            print(f"Epoch {epoch:03d} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            best_model_state = model.state_dict()
            best_params = params
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break
    
final_model = GCN(
    in_channels=5,
    hidden_channels=best_params['h1'],
    out_channels=num_classes,
    activation_fn=activation_map[best_params['activations']]
).to(device)

final_model.load_state_dict(best_model_state)

test_loader = DataLoader(test_data_list, batch_size=best_params["batch_sizes"])
test_acc = evaluate(test_loader)

print("\nBest hyperparameters:", best_params)
print(f"Final Test Accuracy after early stopping: {test_acc:.4f}")

Combination 1/108: {'h1': 10, 'activations': 'relu', 'lrs': 0.0001, 'batch_sizes': 128}
Epoch 050 | Loss: 3.1612 | Train Acc: 0.1339 | Val Acc: 0.1214
Epoch 100 | Loss: 2.8624 | Train Acc: 0.1905 | Val Acc: 0.1548
Epoch 150 | Loss: 2.5601 | Train Acc: 0.2173 | Val Acc: 0.1500
Epoch 200 | Loss: 2.2830 | Train Acc: 0.2268 | Val Acc: 0.1595
Epoch 250 | Loss: 2.0490 | Train Acc: 0.3351 | Val Acc: 0.2952
Epoch 300 | Loss: 1.8757 | Train Acc: 0.3690 | Val Acc: 0.3381
Epoch 350 | Loss: 1.7286 | Train Acc: 0.4220 | Val Acc: 0.3881
Epoch 400 | Loss: 1.6512 | Train Acc: 0.4833 | Val Acc: 0.4500
Epoch 450 | Loss: 1.6200 | Train Acc: 0.5173 | Val Acc: 0.5214
Epoch 500 | Loss: 1.4632 | Train Acc: 0.5417 | Val Acc: 0.5429
Epoch 550 | Loss: 1.3911 | Train Acc: 0.5577 | Val Acc: 0.5548
Epoch 600 | Loss: 1.3442 | Train Acc: 0.5905 | Val Acc: 0.5929
Epoch 650 | Loss: 1.2972 | Train Acc: 0.6077 | Val Acc: 0.6071
Epoch 700 | Loss: 1.2382 | Train Acc: 0.6190 | Val Acc: 0.6119
Epoch 750 | Loss: 1.1937 | Tra